# 01 — Tokenizer

Train a **Byte-Level BPE (Byte Pair Encoding)** tokenizer for A.N.T.H.O.R.

Why byte-level? — works with any text, no unknown tokens, compact vocabulary.

> Runs on **Google Colab** (free T4 tier).
> Uses `tokenizers` library (Rust backend, extremely fast).

## 1. Install dependencies

In [ ]:
!pip install -q tokenizers datasets transformers

## 2. Mount Google Drive (optional)

Run this if you want to save the tokenizer to your Drive for persistence.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Configuration

Tweak these parameters to control vocabulary size & training data.

In [ ]:
# ─── Tokenizer config ───
VOCAB_SIZE   = 16_384       # Small vocab = lighter model
MIN_FREQUENCY = 2           # Drop merges occurring < 2 times
MAX_CHUNK_SIZE = 200_000    # Bytes per chunk (memory safe)

# ─── Data config ───
DATASET_NAME  = "HuggingFaceFW/fineweb-edu"  # High-quality educational data
DATASET_SPLIT = "sample-10BT"                # 10B tokens sample
SAMPLE_SIZE   = 5_000       # Number of documents to use (keep fast for Colab)

# ─── Paths ───
TOKENIZER_PATH = "anthor_tokenizer.json"      # Local save
DRIVE_PATH     = "/content/drive/MyDrive/ANTHOR/tokenizer.json"  # Drive backup

## 4. Load dataset

We use a small slice of **FineWeb-Edu** — high-quality educational web text.
Adjust `SAMPLE_SIZE` above if you want more (or less) data.

In [ ]:
from datasets import load_dataset

print(f"Loading {DATASET_NAME} ({DATASET_SPLIT})...")
ds = load_dataset(DATASET_NAME, split=DATASET_SPLIT, streaming=True)

docs = []
for i, example in enumerate(ds):
    if i >= SAMPLE_SIZE:
        break
    docs.append(example["text"])
    if (i + 1) % 1000 == 0:
        print(f"  Loaded {i + 1}/{SAMPLE_SIZE} documents")

total_bytes = sum(len(d.encode("utf-8")) for d in docs)
print(f"\nDone! {len(docs)} documents, ~{total_bytes / 1e6:.1f} MB of text")

## 5. Train Byte-Level BPE tokenizer

We use HuggingFace `tokenizers` library — it's written in Rust and can process
hundreds of MB per second even on a single CPU.

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, normalizers, decoders
import time

# ── Build tokenizer ──
tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))

# Normalisation: NFC Unicode + lowercase (optional: set to False for case-sensitive)
tokenizer.normalizer = normalizers.Sequence([
    normalizers.NFC(),
    # normalizers.Lowercase(),  # uncomment if you want case-insensitive
])

# Pre-tokenizer: byte-level (no unknown chars, compatible with any input)
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=True)

# Decoder
tokenizer.decoder = decoders.ByteLevel()

# ── Trainer ──
trainer = trainers.BpeTrainer(
    vocab_size=VOCAB_SIZE,
    min_frequency=MIN_FREQUENCY,
    special_tokens=["<pad>", "<unk>", "<s>", "</s>", "<mask>"],
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
)

# ── Train ──
print(f"Training BPE tokenizer (vocab_size={VOCAB_SIZE})...")
t0 = time.time()
tokenizer.train_from_iterator(docs, trainer)
elapsed = time.time() - t0

vocab_size = tokenizer.get_vocab_size()
print(f"Done! {elapsed:.1f}s | vocab_size={vocab_size}")

## 6. Test the tokenizer

In [ ]:
test_texts = [
    "Hello, world! This is ANTHOR.",
    "Attention is all you need.",
    "The quick brown fox jumps over the lazy dog.",
    "Lập trình viên Việt Nam đang chinh phục AI.",
    "123 + 456 = 579",
]

print(f"{'Text':<55} {'Tokens':<20} {'Decoded'}")
print("-" * 110)
for t in test_texts:
    enc = tokenizer.encode(t)
    dec = tokenizer.decode(enc.ids)
    tokens_str = str(enc.ids[:15])
    if len(enc.ids) > 15:
        tokens_str = tokens_str[:-1] + f", ...]"
    print(f"{t:<55} {tokens_str:<20} {dec}")

## 7. Tokenizer statistics

In [ ]:
import numpy as np

# Encode all docs & collect stats
all_token_counts = []
for doc in docs[:100]:  # sample 100 docs for quick stats
    enc = tokenizer.encode(doc)
    all_token_counts.append(len(enc.ids))

all_tokens = np.array(all_token_counts)
total_chars = sum(len(d) for d in docs[:100])

print(f"Statistics (sample=100 docs):")
print(f"  Tokens per doc: mean={all_tokens.mean():.0f}, min={all_tokens.min()}, max={all_tokens.max()}")
print(f"  Compression ratio: {total_chars / all_tokens.sum():.2f} chars/token")
print(f"  Vocabulary size:   {tokenizer.get_vocab_size()}")
print(f"  Total docs:        {len(docs)}")

# Top 10 most frequent tokens
print("\nTop 10 most common tokens:")
ids = []
for doc in docs[:100]:
    ids.extend(tokenizer.encode(doc).ids)
from collections import Counter
freq = Counter(ids).most_common(10)
for token_id, count in freq:
    decoded = tokenizer.decode([token_id])
    print(f"  [{token_id:>5}] {repr(decoded):<20} × {count}")

## 8. Save tokenizer

Saves locally and optionally to Google Drive.

In [ ]:
# Local save
tokenizer.save(TOKENIZER_PATH)
print(f"Saved locally: {TOKENIZER_PATH}")

# Verify
import os
size_kb = os.path.getsize(TOKENIZER_PATH) / 1024
print(f"  File size: {size_kb:.1f} KB")

In [ ]:
# Optional: save to Google Drive
import os
drive_mounted = os.path.exists("/content/drive/MyDrive/")

if drive_mounted:
    os.makedirs(os.path.dirname(DRIVE_PATH), exist_ok=True)
    tokenizer.save(DRIVE_PATH)
    print(f"Saved to Drive: {DRIVE_PATH}")
else:
    print("Drive not mounted — skipping.")

## 9. Push to HuggingFace Hub (optional)

Run this if you have a HuggingFace account and want to share the tokenizer.
First run `huggingface-cli login`, or set `HF_TOKEN` env var.

In [ ]:
# from huggingface_hub import HfApi
# api = HfApi()
# repo_id = "AnNhi/anthor-tokenizer"  # change to your HF username
# api.upload_file(
#     path_or_fileobj=TOKENIZER_PATH,
#     path_in_repo="tokenizer.json",
#     repo_id=repo_id,
#     repo_type="model",
# )
# print(f"Uploaded to https://huggingface.co/{repo_id}")

---

## ✅ Next steps

1. Tokenizer is saved → proceed to **`02_data_pipeline.ipynb`**
2. It tokenizes text into `VOCAB_SIZE` tokens
3. Compatible with any UTF-8 input (English, Vietnamese, code, math...)

> **Tip:** If the model needs a different vocab size later, just tweak `VOCAB_SIZE` above and re-run.